In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset
import matplotlib.pyplot as plt
import os
from utility.ffqd_mnist import FFQD_Dataset, label_for_id
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torchvision

import matplotlib.pyplot as plt;
plt.rcParams['figure.dpi'] = 200
plt.rcParams['xtick.color'] = "white"
plt.rcParams['ytick.color'] = "white"

def to_image(x):
    return torch.Tensor(x).view(1, 28, 28)

DATA_PATH='/leonardo_work/tra26_ictpai/hen_forma_data/ffqd_mnist/'
if not os.path.exists(DATA_PATH):
    DATA_PATH='ffqd_mnist/'

training_data = FFQD_Dataset(DATA_PATH, train=True, transform=to_image)
train_dataloader = DataLoader(training_data, batch_size=64)

test_data = FFQD_Dataset(DATA_PATH, transform=to_image)
test_dataloader = DataLoader(test_data, batch_size=64)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

####
# CREDITS: https://avandekleut.github.io/vae/
####

## Encoder

In [ ]:
class Encoder(nn.Module):
    def __init__(self, latent_dims):
        super(Encoder, self).__init__()
        self.linear1 = nn.Linear(784, 512)
        self.linear2 = nn.Linear(512, latent_dims)

    def forward(self, x):
        x = torch.flatten(x, start_dim=1)
        x = F.relu(self.linear1(x))
        return self.linear2(x)

### Decoder


In [ ]:
class Decoder(nn.Module):
    def __init__(self, latent_dims):
        super(Decoder, self).__init__()
        self.linear1 = nn.Linear(latent_dims, 512)
        self.linear2 = nn.Linear(512, 784)

    def forward(self, z):
        z = F.relu(self.linear1(z))
        z = torch.sigmoid(self.linear2(z))
        return z.reshape((-1, 1, 28, 28))

### Plug it all together


In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, latent_dims):
        super(Autoencoder, self).__init__()
        self.encoder = Encoder(latent_dims)
        self.decoder = Decoder(latent_dims)

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)


In [ ]:
def train(autoencoder, data, epochs=10):
    opt = torch.optim.Adam(autoencoder.parameters())
    for epoch in range(epochs):
        print(f"training epoch {epoch}...")
        for x, y in data:
            x = x.to(device) # GPU
            opt.zero_grad()
            x_hat = autoencoder(x)
            loss = ((x - x_hat)**2).sum()
            loss.backward()
            opt.step()
    return autoencoder


In [ ]:
latent_dims = 2
autoencoder = Autoencoder(latent_dims).to(device) # GPU

autoencoder = train(autoencoder, train_dataloader)

In [ ]:
def plot_latent(autoencoder, data, num_batches=100):
    for i, (x, y) in enumerate(data):
        z = autoencoder.encoder(x.to(device))
        z = z.to('cpu').detach().numpy()
        plt.scatter(z[:, 0], z[:, 1], c=torch.argmax(y, dim=1), cmap='tab10')
        if i > num_batches:
            plt.colorbar()
            break

In [ ]:
plot_latent(autoencoder, train_dataloader)

## Reconstruction from Latent Space


In [ ]:
def plot_reconstructed(autoencoder, r0=(-15, 5), r1=(-8, 8), n=12):
    w = 28
    img = np.zeros((n*w, n*w))
    for i, y in enumerate(np.linspace(*r1, n)):
        for j, x in enumerate(np.linspace(*r0, n)):
            z = torch.Tensor([[x, y]]).to(device)
            x_hat = autoencoder.decoder(z)
            x_hat = x_hat.reshape(28, 28).to('cpu').detach().numpy()
            img[(n-1-i)*w:(n-1-i+1)*w, j*w:(j+1)*w] = x_hat
    plt.imshow(img, extent=[*r0, *r1])

In [ ]:
plot_reconstructed(autoencoder)

<pre>




























</pre>


# Variational from here on :)

we need a new Encoder that samples from a Gaussian distribution

In [ ]:
class VariationalEncoder(nn.Module):
    def __init__(self, latent_dims):
        super(VariationalEncoder, self).__init__()
        self.linear1 = nn.Linear(784, 512)
        self.linear2 = nn.Linear(512, latent_dims)
        self.linear3 = nn.Linear(512, latent_dims)
        self.N = torch.distributions.Normal(0, 1)
        if torch.cuda.is_available():
            self.N.loc = self.N.loc.cuda() # hack to get sampling on the GPU
            self.N.scale = self.N.scale.cuda()
        self.kl = 0

    def forward(self, x):
        x = torch.flatten(x, start_dim=1)
        x = F.relu(self.linear1(x))
        mu =  self.linear2(x)
        sigma = torch.exp(self.linear3(x))
        z = mu + sigma*self.N.sample(mu.shape) # Re-Parametrization trick
        self.kl = (sigma**2 + mu**2 - torch.log(sigma) - 1/2).sum()
        return z

### and adding it together


In [ ]:
class VariationalAutoencoder(nn.Module):
    def __init__(self, latent_dims):
        super(VariationalAutoencoder, self).__init__()
        self.encoder = VariationalEncoder(latent_dims)
        self.decoder = Decoder(latent_dims)

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

added the KL term to the objective function:

In [ ]:
def train_vae(autoencoder, data, epochs=10):
    opt = torch.optim.Adam(autoencoder.parameters())
    for epoch in range(epochs):
        print(f"training epoch {epoch}...")
        for x, y in data:
            x = x.to(device) # GPU if needed
            opt.zero_grad()
            x_hat = autoencoder(x)
            loss = ((x - x_hat)**2).sum() + autoencoder.encoder.kl
            loss.backward()
            opt.step()
    return autoencoder

In [ ]:
vae = VariationalAutoencoder(latent_dims).to(device) # GPU
vae = train_vae(vae, train_dataloader)

In [ ]:
plot_latent(vae, train_dataloader)

In [ ]:
plot_reconstructed(vae, r0=(-3, 3), r1=(-3, 3), n=12)

<pre>




























</pre>

## Fake detection ?

In [ ]:
def test_fake(autoencoder, data):
    fig, axs = plt.subplots(3,2, figsize=(6,6))
    axs= axs.flatten()

    z = torch.Tensor([[-2.5, 3]]).to(device)
    # z = torch.Tensor([[0.5, -.5]]).to(device)
    x_hat = autoencoder.decoder(z)
    x_hat_img = x_hat.reshape(28, 28).to('cpu').detach().numpy()

    axs[0].imshow(x_hat_img, cmap='gray')

    example = data[91][0] # 55 bottle
    example_img = example.reshape(28, 28).to('cpu').detach().numpy()
    # print(example.shape)
    axs[1].imshow(example_img, cmap='gray')


    rec_self_diff = autoencoder(x_hat)
    axs[2].imshow(rec_self_diff.reshape(28, 28).to('cpu').detach().numpy(), cmap='gray')
    axs[4].imshow((rec_self_diff - x_hat).reshape(28, 28).to('cpu').detach().numpy(), vmin=-1, vmax=1)
    print(f"euclidean diff sampled: {((rec_self_diff - x_hat)**2).sum()} ")

    rec_train_diff = autoencoder(example.to(device)).cpu()
    axs[5].imshow((rec_train_diff - example).reshape(28, 28).to('cpu').detach().numpy(), vmin=-1, vmax=1)
    axs[3].imshow(rec_train_diff.reshape(28, 28).to('cpu').detach().numpy(), cmap='gray')
    
    print(f"euclidean diff train data: {((rec_train_diff - example)**2).sum()}")

    return x_hat, x_hat_img
_,_ = test_fake(vae, training_data)

In [ ]:
i1 = 3
i2 = 7
x, y = next(train_dataloader.__iter__()) # hack to grab a batch
y = torch.argmax(y, dim=1)
# print (x.shape,y.shape)
x_1 = x[y == i1][1].to(device) # find a 1
x_2 = x[y == i2][0].to(device) # find a 0
print(f"interpolation from {label_for_id(i1)} to {label_for_id(i2)}")

In [ ]:
def interpolate(autoencoder, x_1, x_2, n=12):
    z_1 = autoencoder.encoder(x_1)
    z_2 = autoencoder.encoder(x_2)
    z = torch.stack([z_1 + (z_2 - z_1)*t for t in np.linspace(0, 1, n)])
    interpolate_list = autoencoder.decoder(z)
    interpolate_list = interpolate_list.to('cpu').detach().numpy()

    w = 28
    img = np.zeros((w, n*w))
    for i, x_hat in enumerate(interpolate_list):
        img[:, i*w:(i+1)*w] = x_hat.reshape(28, 28)
    plt.imshow(img)
    plt.xticks([])
    plt.yticks([])

In [ ]:
from PIL import Image

def interpolate_gif(autoencoder, filename, x_1, x_2, n=100):
    z_1 = autoencoder.encoder(x_1)
    z_2 = autoencoder.encoder(x_2)

    z = torch.stack([z_1 + (z_2 - z_1)*t for t in np.linspace(0, 1, n)])

    interpolate_list = autoencoder.decoder(z)
    interpolate_list = interpolate_list.to('cpu').detach().numpy()*255

    images_list = [Image.fromarray(img.reshape(28, 28)).resize((256, 256)) for img in interpolate_list]
    images_list = images_list + images_list[::-1] # loop back beginning

    images_list[0].save(
        f'{filename}.gif',
        save_all=True,
        append_images=images_list[1:],
        loop=1)

In [ ]:
interpolate_gif(vae, "vae", x_1, x_2)
